In [0]:
# Cria um campo de seleção no topo do notebook
dbutils.widgets.dropdown("environment", "DEV", ["DEV", "PROD"])
env = dbutils.widgets.get("environment").lower() # Retorna "dev" ou "prod"

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Phase 1 - Transform and publish
# MAGIC
# MAGIC Normalize hourly arrays and optionally write to PostgreSQL through JDBC.
# MAGIC
# MAGIC Set `JDBC_URL`, `JDBC_USER`, and `JDBC_PASSWORD` in the notebook/job environment.
# MAGIC Never hard-code credentials.

# COMMAND ----------
# Run the ingestion notebook to populate raw_df
%run ./02_ingest_open_meteo.ipynb

from pyspark.sql import functions as F

# If the previous notebook is not attached to the same workflow, replace this
# with a read from a persisted Bronze Delta table.
cities = [
    ("Brasilia", -15.793889, -47.882778),
    ("Sao Paulo", -23.55052, -46.633308),
    ("Rio de Janeiro", -22.906847, -43.172896),
]

# Busca os dados direto da tabela persistida pela camada anterior
#raw_df = spark.read.table(f"{"environment"}to.bronze_db.open_meteo_raw")

# Example expected raw_df shape:
# city, latitude, longitude, hourly.time, hourly.temperature_2m, ...

weather_df = (
    raw_df
    .select(
        "city", "latitude", "longitude",
        F.explode(
            F.arrays_zip(
                "hourly.time",
                "hourly.temperature_2m",
                "hourly.relative_humidity_2m",
                "hourly.wind_speed_10m",
                "hourly.precipitation",
            )
        ).alias("x")
    )
    .select(
        "city", "latitude", "longitude",
        F.to_timestamp("x.time").alias("observation_time"),
        F.col("x.temperature_2m").cast("double").alias("temperature_c"),
        F.col("x.relative_humidity_2m").cast("double").alias("humidity_pct"),
        F.col("x.wind_speed_10m").cast("double").alias("wind_speed_kmh"),
        F.col("x.precipitation").cast("double").alias("precipitation_mm"),
    )
)

#display(weather_df)

# escrever os dados na tabela
weather_df.write.mode("overwrite").saveAsTable(f"{catalog_name}.bronze_db.open_meteo")